# Barren Plateaus

Why "just make the circuit bigger" fails in quantum machine learning: the cost gradient of a random hardware-efficient circuit vanishes exponentially with qubit count, leaving a flat, untrainable landscape.

**Objectives:**
- Compute a cost and its gradient EXACTLY from `statevector(circuit)` using the parameter-shift rule
- Measure how the variance of the gradient scales with qubit count for random circuits
- Contrast a GLOBAL cost (a product observable on all qubits) with a LOCAL cost (Z on one qubit) and watch the global variance collapse faster
- Connect the experiment to the four standard mitigations: local cost, structured ansatz, identity initialization, layerwise training

**Reference:** See [`../GUIDE.md`](../GUIDE.md) for concept explanations and context.
<!-- browser-runnable -->

In [ ]:
# Setup: Run this cell first
# Requires: pip install -e '.[dev]' from the project root (see `make setup`)

from braket.circuits import Circuit
from braket.devices import LocalSimulator
import numpy as np
import matplotlib.pyplot as plt
from lib.utils.statevector import statevector

# Use local simulator by default (free, instant)
device = LocalSimulator()

## 1. A random hardware-efficient ansatz

The circuits that cause barren plateaus are *expressive* and *unstructured*: a stack of single-qubit `RY` rotations (one trainable angle per qubit per layer) followed by a nearest-neighbour entangling ladder of `CZ` gates. This is the generic "hardware-efficient" template from notebook `05-qnn-architecture` — easy to run on real hardware, but with no problem structure to anchor the landscape.

We deliberately span all `n` qubits. In qcsim the state vector only covers the qubits a circuit actually touches, so we add a trailing identity `.i(n - 1)` to force the full length-`2^n` state vector even when the last qubit is otherwise idle.

In [ ]:
def hardware_efficient(n_qubits, n_layers, params):
    """Random hardware-efficient ansatz: (RY rotations + CZ ladder) x n_layers.

    params is a flat array of length n_qubits * n_layers, one angle per
    (layer, qubit). Returns a Circuit spanning all n qubits.
    """
    circuit = Circuit()
    k = 0
    for _ in range(n_layers):
        for q in range(n_qubits):
            circuit.ry(q, params[k])
            k += 1
        for q in range(n_qubits - 1):
            circuit.cz(q, q + 1)
    # Force the state vector to span all n qubits (big-endian, full width).
    circuit.i(n_qubits - 1)
    return circuit


# Sanity check: 3 qubits, 2 layers => 6 trainable angles, full-width state.
_demo = hardware_efficient(3, 2, np.zeros(6))
_sv = np.array(statevector(_demo))
print("qubit_count:", _demo.qubit_count)
print("state vector length:", len(_sv), "(expected", 2 ** 3, ")")
assert len(_sv) == 2 ** 3  # full-width state because of the trailing .i(n-1)
# All angles zero => circuit is identity on |000>, so we stay in |000>.
assert np.isclose(abs(_sv[0]), 1.0)
print("All-zero angles keep us in |000>:", np.isclose(abs(_sv[0]), 1.0))

## 2. Two cost functions: global vs. local

Both costs are expectation values of a **diagonal** observable, so we can read them straight off the state vector without any sampling:

$$\langle O \rangle = \sum_i \text{sign}_i \, |sv[i]|^2$$

- **Global cost** — `Z` on *every* qubit, i.e. the product `Z_0 Z_1 ... Z_{n-1}`. The sign of basis state `i` is `+1` if `i` has even Hamming weight (parity), else `-1`. This observable touches all `n` qubits at once; it is the kind of cost that concentrates exponentially.
- **Local cost** — `Z` on qubit 0 only. In big-endian qcsim, qubit 0 is the top bit, so the sign is `+1` when the top bit is 0 and `-1` when it is 1. A local observable "sees" only a small subsystem and resists the collapse.

We precompute the sign vectors once per qubit count.

In [ ]:
def signs_global(n):
    """Signs for Z on ALL qubits (parity of the bitstring)."""
    s = np.empty(2 ** n)
    for i in range(2 ** n):
        s[i] = 1.0 if bin(i).count("1") % 2 == 0 else -1.0
    return s


def signs_local_z0(n):
    """Signs for Z on qubit 0 only (top/MSB bit in big-endian qcsim)."""
    s = np.empty(2 ** n)
    for i in range(2 ** n):
        top_bit = (i >> (n - 1)) & 1
        s[i] = 1.0 if top_bit == 0 else -1.0
    return s


def expectation(circuit, signs):
    """Exact <O> for a diagonal observable from the state vector."""
    sv = np.array(statevector(circuit))
    probs = np.abs(sv) ** 2
    return float(np.sum(signs * probs))


# Quick check on |+++>: <Z_0 Z_1 Z_2> = 0 and <Z_0> = 0 (unbiased superposition).
_plus = Circuit().h(0).h(1).h(2).i(2)
print("<Z global> on |+++>:", round(expectation(_plus, signs_global(3)), 6))
print("<Z_0   > on |+++>:", round(expectation(_plus, signs_local_z0(3)), 6))
assert abs(expectation(_plus, signs_global(3))) < 1e-9
assert abs(expectation(_plus, signs_local_z0(3))) < 1e-9

## 3. Exact gradient via the parameter-shift rule

For a gate generated by a Pauli rotation (like `RY(theta) = exp(-i theta Y / 2)`), the exact derivative of the cost with respect to that angle is

$$\frac{\partial \langle O \rangle}{\partial \theta} = \tfrac{1}{2}\big[\, \langle O \rangle_{\theta + \pi/2} - \langle O \rangle_{\theta - \pi/2} \,\big].$$

This is not a finite-difference approximation — for a single Pauli generator it is the analytic gradient. We evaluate both shifted circuits exactly from their state vectors, so the gradient we record carries no sampling noise. We always differentiate the **first** parameter; by symmetry of the random ansatz any single angle is representative.

In [ ]:
def grad_param0(n_qubits, n_layers, params, signs):
    """Exact d<O>/d(params[0]) via the parameter-shift rule."""
    shift = np.pi / 2
    plus = params.copy()
    plus[0] += shift
    minus = params.copy()
    minus[0] -= shift
    cost_plus = expectation(hardware_efficient(n_qubits, n_layers, plus), signs)
    cost_minus = expectation(hardware_efficient(n_qubits, n_layers, minus), signs)
    return 0.5 * (cost_plus - cost_minus)


# One concrete gradient at n=3 with a fixed parameter set.
np.random.seed(0)
_p = np.random.uniform(0, 2 * np.pi, size=3 * 2)
print("global-cost grad of param 0:", round(grad_param0(3, 2, _p, signs_global(3)), 6))
print("local-cost  grad of param 0:", round(grad_param0(3, 2, _p, signs_local_z0(3)), 6))

## 4. The plateau: gradient variance vs. qubit count

A single gradient tells us nothing — on a plateau the gradient is not biased, it is just *random with tiny spread*. The diagnostic is the **variance of the gradient across many random initializations**. If that variance shrinks toward zero as we add qubits, an optimizer starting from a random point sees an essentially flat landscape and cannot find a descent direction.

For each `n` in `2..5` we draw `n_samples = 30` random parameter vectors and record the variance of the param-0 gradient, for both the global and the local cost. We reset the seed at the start of each `n` so the global and local costs are evaluated on the **same** random circuits (a fair, paired comparison), and the whole experiment is reproducible.

In [ ]:
n_layers = 2
n_samples = 30
qubit_counts = [2, 3, 4, 5]

var_global = {}
var_local = {}

for n in qubit_counts:
    sg = signs_global(n)
    sl = signs_local_z0(n)
    np.random.seed(0)  # same random circuits for both costs at this n
    grads_global = []
    grads_local = []
    for _ in range(n_samples):
        params = np.random.uniform(0, 2 * np.pi, size=n * n_layers)
        grads_global.append(grad_param0(n, n_layers, params, sg))
        grads_local.append(grad_param0(n, n_layers, params, sl))
    var_global[n] = float(np.var(grads_global))
    var_local[n] = float(np.var(grads_local))
    print(
        f"n={n}  Var[grad] global = {var_global[n]:.5e}   local = {var_local[n]:.5e}"
    )

## 5. Self-verifying asserts

Two deterministic facts (with `seed=0`) make the barren-plateau story concrete:

1. The **global** gradient variance at the largest `n` is far smaller than at the smallest `n` — it shrinks as we add qubits. That is the plateau forming.
2. At the largest `n`, the **local** cost has a larger gradient variance than the **global** cost — measuring fewer qubits keeps the landscape trainable.

The thresholds below are loose but comfortably satisfied (the global variance falls by roughly an order of magnitude across the sweep).

In [ ]:
n_small, n_large = qubit_counts[0], qubit_counts[-1]

ratio = var_global[n_small] / var_global[n_large]
print(f"global Var[grad]: n={n_small} -> {var_global[n_small]:.5e}")
print(f"global Var[grad]: n={n_large} -> {var_global[n_large]:.5e}")
print(f"shrink factor (small / large): {ratio:.2f}x")
print(f"local  Var[grad] at n={n_large}: {var_local[n_large]:.5e}")

# Plateau forming: global variance collapses with qubit count.
assert var_global[n_large] < 0.5 * var_global[n_small], "global variance should shrink with n"

# Local cost is the mitigation: larger gradient variance than the global cost.
assert var_local[n_large] > var_global[n_large], "local cost should keep larger gradients"

print("\nAsserts passed: global cost plateaus, local cost stays trainable.")

## 6. Visualize the collapse

A semilog plot makes the contrast obvious: the global-cost variance trends sharply downward with qubit count (the plateau), while the local-cost variance stays high. Real barren-plateau curves are exponential (`Var ~ 2^-n`); at these tiny qubit counts we see the onset of that decay rather than the full asymptotic slope, but the qualitative gap between global and local is already clear.

In [ ]:
ns = np.array(qubit_counts)
gvals = np.array([var_global[n] for n in qubit_counts])
lvals = np.array([var_local[n] for n in qubit_counts])

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.semilogy(ns, gvals, "o-", color="#c0392b", label="global cost (Z on all qubits)")
ax.semilogy(ns, lvals, "s-", color="#2471a3", label="local cost (Z on qubit 0)")
ax.set_xlabel("number of qubits n")
ax.set_ylabel("Var[ d cost / d theta_0 ]  (log scale)")
ax.set_title("Gradient variance vs. qubit count")
ax.set_xticks(ns)
ax.grid(True, which="both", alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## 7. "Now measure it": a plateau gradient is hidden by shot noise

The variance above is computed exactly from the state vector. On real hardware you only get samples, and the punchline is brutal: once the true gradient is smaller than the shot noise of your estimator, you cannot even *detect* a descent direction without an exponential number of shots.

Here we take one plateau-y circuit at `n = 5`, measure its global-cost expectation with a modest shot budget, and compare the sampled value to the exact one. The sampling error (order `1/sqrt(shots)`) easily swamps the tiny gradients we measured above.

In [ ]:
n = 5
np.random.seed(1)
params = np.random.uniform(0, 2 * np.pi, size=n * n_layers)
circ = hardware_efficient(n, n_layers, params)
sg = signs_global(n)

# Exact global-cost expectation from the state vector.
exact = expectation(circ, sg)

# Sampled estimate: <Z_global> = sum over outcomes of parity(bitstring) * prob.
shots = 2000
result = device.run(circ, shots=shots).result()
counts = result.measurement_counts
est = 0.0
for bitstring, c in counts.items():
    parity = 1.0 if bitstring.count("1") % 2 == 0 else -1.0
    est += parity * (c / shots)

print(f"exact   <Z global> = {exact:+.5f}")
print(f"sampled <Z global> = {est:+.5f}  ({shots} shots)")
print(f"sampling error     = {abs(est - exact):.5f}")
print(f"approx shot noise  ~ 1/sqrt(shots) = {1.0 / np.sqrt(shots):.5f}")
print("\nA plateau gradient (<~1e-2) is buried under this noise floor.")

## 8. Mitigations

The experiment above already demonstrates the single most important fix; the rest are how practitioners keep deep QML models trainable:

- **Local cost functions** *(demonstrated)* — measure a few qubits, not all of them. Cerezo et al. proved local costs have at-worst-polynomially-vanishing gradients for shallow circuits, versus exponential for global costs. This is exactly the gap you just plotted.
- **Structured / problem-inspired ansatze** — replace the generic hardware-efficient block with a circuit that encodes known structure (e.g. a Hamiltonian variational ansatz, or symmetry-preserving gates). Less expressivity in the wrong directions means a less flat landscape.
- **Identity (small-angle) initialization** — start every block near the identity (angles near 0, or paired blocks that cancel) so the initial circuit is shallow-effective and the gradient is non-trivial; let depth grow only as training progresses.
- **Layerwise / layer-by-layer training** — train one or a few layers at a time, freezing the rest, so the optimizer never faces the full deep landscape at once.

All four share one idea: a barren plateau comes from too much unstructured expressivity evaluated globally. Reduce the effective expressivity, the locality of the cost, or both.

## Exercises

In [ ]:
# Exercise 1: Locality knob.
# Define a cost that measures Z on the first k qubits (k = 1 is fully local,
# k = n is the global cost). Sweep k from 1 to n at n = 5 and record the
# gradient variance for each k. Where does the plateau start to bite?
#
# def signs_first_k(n, k):
#     s = np.empty(2 ** n)
#     for i in range(2 ** n):
#         # parity over only the top k bits (qubits 0..k-1 in big-endian)
#         top_k = i >> (n - k)
#         s[i] = 1.0 if bin(top_k).count("1") % 2 == 0 else -1.0
#     return s
#
# TODO: loop k in range(1, n + 1), reuse grad_param0 with signs_first_k(n, k),
# TODO: collect Var[grad] over 30 random seeds, and plot variance vs. k.

In [ ]:
# Exercise 2: Depth makes it worse.
# Barren plateaus deepen with circuit DEPTH as well as width. Fix n = 4 and
# the global cost, then sweep n_layers in [1, 2, 4, 8]. Record the gradient
# variance for each depth.
#
# TODO: for L in [1, 2, 4, 8]:
# TODO:     np.random.seed(0)
# TODO:     draw 30 random params of length n * L, compute grad_param0(n, L, ...)
# TODO:     store np.var(grads)
# TODO: plot Var[grad] vs. L on a log y-axis. Does deeper help or hurt?

## Summary

- A **barren plateau** is a landscape where the cost gradient has vanishingly small variance across random initializations — not biased, just flat — so an optimizer cannot find a descent direction.
- We computed costs and their **exact** parameter-shift gradients straight from `statevector(circuit)`, with no sampling noise, for random hardware-efficient circuits.
- The **global** cost (a product `Z` on all qubits) showed its gradient variance collapsing as qubits were added, while the **local** cost (`Z` on one qubit) kept a much larger variance — the asserts pin both facts down deterministically.
- A "now measure it" sampling run showed that on real hardware **shot noise buries** a plateau gradient, which is why the plateau is fatal in practice.
- Mitigations all reduce unstructured expressivity or cost locality: **local costs** (demonstrated), **structured ansatze**, **identity initialization**, and **layerwise training**.

**Next:** [`07-hybrid-ml-job.ipynb`](07-hybrid-ml-job.ipynb) — package a trainable quantum model as a hybrid quantum-classical job.